# 자율 실험실 실습

**Self-driving Laboratory · SDL · 자율실험**

자동화 장비와 데이터 기반 의사결정을 결합해 다음 실험을 선택하고 수행하는 시스템.

소재 분야에서 이해하기: 로봇 합성 결과를 AI가 분석해 다음 실험 조건을 정한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [자율 소재 실험실 A-Lab 연구](https://www.nature.com/articles/s41586-023-06734-w)

## 1. 장비 제약과 실패가 있는 루프

자율 실험실은 제안만이 아니라 장비 제약·실패·재시도까지 다뤄야 합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.stats import norm

FEASIBLE_TEMPERATURE = (0.2, 0.9)     # 장비가 낼 수 있는 범위
FAILURE_RATE = 0.15                   # 합성 실패 확률
BATCH = 4                             # 한 번에 4개 시료 동시 처리

def synthesise(point, local):
    if local.random() < FAILURE_RATE:
        return None                   # 실패: 결과 없음
    value = 88 * np.exp(-((point[0] - 0.7) ** 2 + (point[1] - 0.4) ** 2) / 0.05)
    return value + local.normal(0, 2.0)

grid_a, grid_b = np.meshgrid(np.linspace(*FEASIBLE_TEMPERATURE, 50), np.linspace(0, 1, 50))
candidates = np.column_stack([grid_a.ravel(), grid_b.ravel()])
print('장비 제약을 반영한 후보 %d점, 배치 크기 %d, 실패율 %.0f%%'
      % (len(candidates), BATCH, 100 * FAILURE_RATE))

In [ ]:
local = np.random.default_rng(0)
seen_x, seen_y, failures = [], [], 0
for point in [[0.3, 0.2], [0.8, 0.8], [0.5, 0.5], [0.25, 0.75]]:
    result = synthesise(np.array(point), local)
    if result is None:
        failures += 1
    else:
        seen_x.append(point); seen_y.append(result)

history, attempts = [], 0
for cycle in range(1, 9):
    model = GaussianProcessRegressor(kernel=ConstantKernel(50.0) * RBF([0.2, 0.2]),
                                     normalize_y=True, alpha=4.0, random_state=0)
    model.fit(np.array(seen_x), seen_y)
    mean, std = model.predict(candidates, return_std=True)
    best = max(seen_y)
    z = (mean - best) / np.maximum(std, 1e-9)
    acquisition = (mean - best) * norm.cdf(z) + std * norm.pdf(z)
    # 배치 선택: 최고점을 고르고 그 주변을 눌러 다양성을 확보
    batch = []
    scores = acquisition.copy()
    for _ in range(BATCH):
        index = int(np.argmax(scores))
        batch.append(candidates[index])
        scores -= acquisition[index] * np.exp(-((candidates - candidates[index]) ** 2).sum(1) / 0.02)
    for point in batch:
        attempts += 1
        result = synthesise(point, local)
        if result is None:
            failures += 1
            continue
        seen_x.append(list(point)); seen_y.append(result)
    history.append(max(seen_y))
    print('cycle %d: 시도 %d건, 누적 실패 %d건, 최고 수율 %.1f%%' % (cycle, BATCH, failures, history[-1]))
print('\n총 시도 %d건 중 실패 %d건(%.0f%%), 유효 데이터 %d건'
      % (attempts + 4, failures, 100 * failures / (attempts + 4), len(seen_y)))

In [ ]:
points = np.array(seen_x)
plt.contourf(grid_a, grid_b, model.predict(candidates).reshape(grid_a.shape), levels=20, cmap='viridis')
plt.scatter(points[:, 0], points[:, 1], c='red', s=20, label='successful runs')
plt.scatter([0.7], [0.4], marker='*', c='white', s=160, label='true optimum')
plt.axvline(FEASIBLE_TEMPERATURE[0], color='w', ls=':'); plt.axvline(FEASIBLE_TEMPERATURE[1], color='w', ls=':')
plt.xlabel('condition a (equipment limited)'); plt.ylabel('condition b'); plt.legend(fontsize=8); plt.show()
plt.plot(history, 'o-'); plt.xlabel('cycle'); plt.ylabel('best yield (%)'); plt.show()
print('실패한 시도는 데이터가 되지 않으므로 예산 계획에 실패율을 반영해야 합니다.')
print('또 장비 제약이 참 최적점을 포함하지 않으면, 루프는 제약 안에서의 최적만 찾습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#self-driving-lab)을 여세요.